# Customer Churn Prediction using XGBoost and Random Forest

This notebook analyzes the Telco Customer Churn dataset and builds predictive models to identify customers likely to churn.

## Table of Contents
1. [Data Loading and Overview](#1.-Data-Loading-and-Overview)
2. [Exploratory Data Analysis (EDA)](#2.-Exploratory-Data-Analysis)
3. [Feature Engineering](#3.-Feature-Engineering)
4. [Model Training](#4.-Model-Training)
   - XGBoost
   - Random Forest
5. [Model Evaluation](#5.-Model-Evaluation)
6. [Customer Segmentation with K-means](#6.-Customer-Segmentation-with-K-means)

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.cluster import KMeans
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Data Loading and Overview

In [ ]:
# Load the dataset
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Basic statistics
print("Dataset Info:")
df.info()
print("\n" + "="*50)
print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("\nChurn Distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn Rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.2f}%")

## 2. Exploratory Data Analysis

In [ ]:
# Data preprocessing - fix TotalCharges column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [ ]:
# Visualization 1: Churn Distribution and Key Features
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Customer Churn Analysis - Key Insights', fontsize=16, fontweight='bold')

# 1. Churn Distribution
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0, 0].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', 
               colors=colors, startangle=90, explode=(0, 0.1))
axes[0, 0].set_title('Overall Churn Distribution', fontweight='bold')

# 2. Churn by Contract Type
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', ax=axes[0, 1], color=colors)
axes[0, 1].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0, 1].set_xlabel('Contract Type')
axes[0, 1].set_ylabel('Percentage (%)')
axes[0, 1].legend(['No Churn', 'Churn'])
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Churn by Internet Service
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
internet_churn.plot(kind='bar', ax=axes[0, 2], color=colors)
axes[0, 2].set_title('Churn Rate by Internet Service', fontweight='bold')
axes[0, 2].set_xlabel('Internet Service')
axes[0, 2].set_ylabel('Percentage (%)')
axes[0, 2].legend(['No Churn', 'Churn'])
axes[0, 2].tick_params(axis='x', rotation=45)

# 4. Tenure Distribution by Churn
df[df['Churn']=='No']['tenure'].hist(ax=axes[1, 0], bins=30, alpha=0.7, label='No Churn', color='#2ecc71')
df[df['Churn']=='Yes']['tenure'].hist(ax=axes[1, 0], bins=30, alpha=0.7, label='Churn', color='#e74c3c')
axes[1, 0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# 5. Monthly Charges Distribution by Churn
df[df['Churn']=='No']['MonthlyCharges'].hist(ax=axes[1, 1], bins=30, alpha=0.7, label='No Churn', color='#2ecc71')
df[df['Churn']=='Yes']['MonthlyCharges'].hist(ax=axes[1, 1], bins=30, alpha=0.7, label='Churn', color='#e74c3c')
axes[1, 1].set_title('Monthly Charges Distribution by Churn', fontweight='bold')
axes[1, 1].set_xlabel('Monthly Charges ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

# 6. Churn by Senior Citizen Status
senior_churn = pd.crosstab(df['SeniorCitizen'], df['Churn'], normalize='index') * 100
senior_churn.plot(kind='bar', ax=axes[1, 2], color=colors)
axes[1, 2].set_title('Churn Rate by Senior Citizen Status', fontweight='bold')
axes[1, 2].set_xlabel('Senior Citizen (0=No, 1=Yes)')
axes[1, 2].set_ylabel('Percentage (%)')
axes[1, 2].legend(['No Churn', 'Churn'])
axes[1, 2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('images/churn_analysis_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: images/churn_analysis_overview.png")

In [ ]:
# Additional insights
print("Key Insights from EDA:")
print("="*60)
print(f"1. Overall Churn Rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.2f}%")
print(f"\n2. Churn Rate by Contract:")
print(df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()*100).round(2))
print(f"\n3. Average Tenure by Churn:")
print(df.groupby('Churn')['tenure'].mean().round(2))
print(f"\n4. Average Monthly Charges by Churn:")
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))
print(f"\n5. Senior Citizen Churn Rate:")
print(df.groupby('SeniorCitizen')['Churn'].apply(lambda x: (x=='Yes').mean()*100).round(2))

## 3. Feature Engineering

In [ ]:
# Create a copy for feature engineering
df_model = df.copy()

# Drop customerID as it's not useful for prediction
df_model = df_model.drop('customerID', axis=1)

# Create new features
# Handle zero tenure by replacing with 1 to avoid division issues
df_model['ChargePerMonth'] = df_model['TotalCharges'] / df_model['tenure'].replace(0, 1)
df_model['TenureGroup'] = pd.cut(df_model['tenure'], bins=[0, 12, 24, 48, 100], 
                                  labels=['0-1 year', '1-2 years', '2-4 years', '4+ years'])

print("New features created:")
print("- ChargePerMonth: Average charge per month")
print("- TenureGroup: Categorized tenure")
print("\nFeature statistics:")
print(df_model[['ChargePerMonth', 'TenureGroup']].describe())

In [ ]:
# Encode categorical variables
le = LabelEncoder()

# Binary encoding for Yes/No columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Label encoding for other categorical columns
categorical_cols = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                   'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
                   'StreamingMovies', 'Contract', 'PaymentMethod', 'TenureGroup']

for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Categorical encoding completed.")
print(f"\nFinal dataset shape: {df_model.shape}")
print(f"Features: {df_model.columns.tolist()}")

In [ ]:
# Prepare features and target
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

## 4. Model Training

### 4.1 XGBoost Classifier

In [ ]:
# Train XGBoost model
print("Training XGBoost model...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("✓ XGBoost model trained successfully!")
print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=['No Churn', 'Churn']))

### 4.2 Random Forest Classifier

In [ ]:
# Train Random Forest model
print("Training Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print("✓ Random Forest model trained successfully!")
print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred, target_names=['No Churn', 'Churn']))

## 5. Model Evaluation

### ROC Curve and AUC Comparison

In [ ]:
# Visualization 2: Model Performance Comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Model Performance Evaluation', fontsize=16, fontweight='bold')

# 1. ROC Curves
# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_pred_proba)
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)

# Random Forest ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_pred_proba)
roc_auc_rf = auc(fpr_rf, tpr_rf)

axes[0].plot(fpr_xgb, tpr_xgb, color='#3498db', lw=2.5, 
             label=f'XGBoost (AUC = {roc_auc_xgb:.3f})')
axes[0].plot(fpr_rf, tpr_rf, color='#e74c3c', lw=2.5, 
             label=f'Random Forest (AUC = {roc_auc_rf:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curves - Model Comparison', fontweight='bold', fontsize=12)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# 2. Confusion Matrix - XGBoost
cm_xgb = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[1].set_ylabel('Actual', fontsize=11)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_title(f'XGBoost Confusion Matrix (AUC={roc_auc_xgb:.3f})', fontweight='bold', fontsize=12)

# 3. Confusion Matrix - Random Forest
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Reds', ax=axes[2],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[2].set_ylabel('Actual', fontsize=11)
axes[2].set_xlabel('Predicted', fontsize=11)
axes[2].set_title(f'Random Forest Confusion Matrix (AUC={roc_auc_rf:.3f})', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('images/model_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: images/model_performance_comparison.png")

In [ ]:
# Feature Importance Analysis
feature_importance_xgb = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

feature_importance_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

print("Top 10 Features - XGBoost:")
print(feature_importance_xgb.to_string(index=False))
print("\n" + "="*60)
print("\nTop 10 Features - Random Forest:")
print(feature_importance_rf.to_string(index=False))

In [ ]:
# Model comparison summary
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\nXGBoost:")
print(f"  - AUC-ROC Score: {roc_auc_xgb:.4f}")
print(f"  - Accuracy: {(xgb_pred == y_test).mean():.4f}")

print(f"\nRandom Forest:")
print(f"  - AUC-ROC Score: {roc_auc_rf:.4f}")
print(f"  - Accuracy: {(rf_pred == y_test).mean():.4f}")

print("\n" + "="*60)
if roc_auc_xgb > roc_auc_rf:
    print("🏆 Winner: XGBoost performs better with higher AUC-ROC score!")
else:
    print("🏆 Winner: Random Forest performs better with higher AUC-ROC score!")
print("="*60)

## 6. Customer Segmentation with K-means

Using K-means clustering to segment customers based on their characteristics.

In [ ]:
# Prepare data for clustering (using numerical features)
clustering_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ChargePerMonth']
X_cluster = df_model[clustering_features].copy()

# Scale the features for clustering
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Find optimal number of clusters using elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)

print("Calculating optimal number of clusters...")
print("✓ Elbow method analysis complete")

In [ ]:
# Apply K-means with optimal clusters (k=4)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_model['Cluster'] = kmeans.fit_predict(X_cluster_scaled)
df['Cluster'] = df_model['Cluster']

print(f"K-means clustering with k={optimal_k} completed.")
print(f"\nCluster distribution:")
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Visualization 3: Customer Segmentation
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Segmentation Analysis with K-means Clustering', fontsize=16, fontweight='bold')

# 1. Elbow Method
axes[0, 0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].axvline(x=optimal_k, color='r', linestyle='--', linewidth=2, label=f'Optimal k={optimal_k}')
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0, 0].set_ylabel('Inertia', fontsize=11)
axes[0, 0].set_title('Elbow Method for Optimal K', fontweight='bold', fontsize=12)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. Cluster distribution by Tenure and Monthly Charges
scatter = axes[0, 1].scatter(df['tenure'], df['MonthlyCharges'], 
                            c=df['Cluster'], cmap='viridis', 
                            alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
axes[0, 1].set_xlabel('Tenure (months)', fontsize=11)
axes[0, 1].set_ylabel('Monthly Charges ($)', fontsize=11)
axes[0, 1].set_title('Customer Segments: Tenure vs Monthly Charges', fontweight='bold', fontsize=12)
cbar = plt.colorbar(scatter, ax=axes[0, 1])
cbar.set_label('Cluster', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# 3. Churn rate by cluster
cluster_churn = df.groupby('Cluster')['Churn'].apply(lambda x: (x=='Yes').mean() * 100)
bars = axes[1, 0].bar(cluster_churn.index, cluster_churn.values, 
                     color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'], 
                     edgecolor='black', linewidth=1.5)
axes[1, 0].set_xlabel('Cluster', fontsize=11)
axes[1, 0].set_ylabel('Churn Rate (%)', fontsize=11)
axes[1, 0].set_title('Churn Rate by Customer Segment', fontweight='bold', fontsize=12)
axes[1, 0].set_xticks(cluster_churn.index)
# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Cluster characteristics heatmap
cluster_summary = df.groupby('Cluster')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean()
cluster_summary_normalized = (cluster_summary - cluster_summary.mean()) / cluster_summary.std()
sns.heatmap(cluster_summary_normalized.T, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=0, ax=axes[1, 1], cbar_kws={'label': 'Normalized Value'})
axes[1, 1].set_xlabel('Cluster', fontsize=11)
axes[1, 1].set_ylabel('Feature', fontsize=11)
axes[1, 1].set_title('Normalized Cluster Characteristics', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('images/customer_segmentation_kmeans.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: images/customer_segmentation_kmeans.png")

In [ ]:
# Analyze cluster characteristics
print("\n" + "="*70)
print("CUSTOMER SEGMENT ANALYSIS")
print("="*70)

for cluster_id in sorted(df['Cluster'].unique()):
    cluster_data = df[df['Cluster'] == cluster_id]
    churn_rate = (cluster_data['Churn'] == 'Yes').mean() * 100
    
    print(f"\n📊 Cluster {cluster_id}:")
    print(f"   Size: {len(cluster_data)} customers ({len(cluster_data)/len(df)*100:.1f}%)")
    print(f"   Churn Rate: {churn_rate:.2f}%")
    print(f"   Avg Tenure: {cluster_data['tenure'].mean():.1f} months")
    print(f"   Avg Monthly Charges: ${cluster_data['MonthlyCharges'].mean():.2f}")
    print(f"   Avg Total Charges: ${cluster_data['TotalCharges'].mean():.2f}")
    
    # Classify the segment
    if churn_rate > 30:
        risk_level = "🔴 HIGH RISK"
    elif churn_rate > 20:
        risk_level = "🟡 MEDIUM RISK"
    else:
        risk_level = "🟢 LOW RISK"
    print(f"   Risk Level: {risk_level}")

print("\n" + "="*70)

## Summary and Conclusions

### Key Findings:

1. **Churn Patterns:**
   - Customers with month-to-month contracts have significantly higher churn rates
   - Newer customers (low tenure) are more likely to churn
   - Higher monthly charges correlate with increased churn risk
   - Senior citizens show higher churn rates

2. **Model Performance:**
   - Both XGBoost and Random Forest models perform well
   - AUC-ROC scores indicate good predictive capability
   - Key predictive features include tenure, contract type, and monthly charges

3. **Customer Segmentation:**
   - K-means clustering identified distinct customer segments
   - Different segments show varying churn rates
   - High-risk segments can be targeted for retention strategies

### Recommendations:

1. **Retention Strategy:**
   - Focus on customers in the first 12 months of tenure
   - Encourage transition from month-to-month to longer contracts
   - Provide incentives for customers with high monthly charges

2. **Targeted Interventions:**
   - Implement early warning system using the predictive models
   - Customize retention offers based on customer segment
   - Monitor high-risk clusters closely

3. **Service Improvements:**
   - Enhance customer onboarding experience
   - Improve technical support services
   - Consider pricing strategies for different customer segments